# Laboratorio 2 — Complejidad y búsqueda de hiperparámetros (AlpesHearth)

**Objetivo:** comparar regresión polinomial, regresión regularizada (Ridge/Lasso) y regresión polinomial regularizada; seleccionar el mejor modelo por desempeño y estabilidad (CV); y estimar intervalos de confianza (bootstrap) en test.

> **Notas importantes**
- Este notebook es una **plantilla completa** con todas las actividades del laboratorio, pero debes adaptar:
  - la ruta del dataset limpio,
  - el nombre de la variable objetivo (`TARGET_COL`),
  - (si aplica) columnas a excluir (IDs, leakage, etc.).
- Mantén el **test set congelado**: no lo uses para ajustar hiperparámetros.

## 0. Setup del entorno

In [ ]:
# ======================
# 0) Imports y settings
# ======================

import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

## 1. Funciones auxiliares (métricas, tablas, gráficas, bootstrap)

In [ ]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

def summarize_gridsearch(gs: GridSearchCV, sort_by: str = "mean_test_RMSE") -> pd.DataFrame:
    """Convierte cv_results_ de GridSearchCV en tabla amigable.

    Nota: usaremos scoring con nombres personalizados (ver sección 4).
    """
    results = pd.DataFrame(gs.cv_results_)
    # Ordenar
    if sort_by in results.columns:
        results = results.sort_values(sort_by, ascending=True)
    # Selección de columnas relevantes
    keep = [c for c in results.columns if c.startswith("param_")] + [
        "mean_test_RMSE", "std_test_RMSE",
        "mean_test_MAE",  "std_test_MAE",
        "mean_test_R2",   "std_test_R2",
        "rank_test_RMSE"
    ]
    keep = [c for c in keep if c in results.columns]
    return results[keep].reset_index(drop=True)

def plot_validation_curve_from_table(df: pd.DataFrame, x_param_col: str, title: str):
    """Grafica mean±std de RMSE/MAE/R2 vs un hiperparámetro discreto (p.ej. degree)."""
    x = df[x_param_col].astype(float).values

    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.set_title(title)
    ax.set_xlabel(x_param_col.replace("param_", ""))

    # RMSE
    y = df["mean_test_RMSE"].values
    ystd = df["std_test_RMSE"].values
    ax.plot(x, y, marker="o", label="CV mean RMSE")
    ax.fill_between(x, y - ystd, y + ystd, alpha=0.2, label="±1 std (RMSE)")

    ax.set_ylabel("RMSE (CV)")
    ax.legend()
    plt.show()

def bootstrap_ci_metrics(model, X_test: pd.DataFrame, y_test: pd.Series,
                         n_boot: int = 500, ci: float = 0.95) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Bootstrap en test set para estimar IC de métricas.

    Retorna:
      - df_samples: métricas por remuestreo
      - df_ci: tabla con IC inferior/superior y media
    """
    rng = np.random.default_rng(RANDOM_STATE)
    n = len(y_test)
    metrics_list = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)  # remuestreo con reemplazo
        Xb = X_test.iloc[idx]
        yb = y_test.iloc[idx]
        pred = model.predict(Xb)
        metrics_list.append(compute_metrics(yb, pred))

    df_samples = pd.DataFrame(metrics_list)

    alpha = (1 - ci) / 2
    lower = df_samples.quantile(alpha)
    upper = df_samples.quantile(1 - alpha)

    df_ci = pd.DataFrame({
        "mean": df_samples.mean(),
        f"ci_{int(ci*100)}_lower": lower,
        f"ci_{int(ci*100)}_upper": upper,
    })

    return df_samples, df_ci

def plot_bootstrap_distributions(df_samples: pd.DataFrame, bins: int = 30):
    for col in df_samples.columns:
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.set_title(f"Bootstrap distribution — {col}")
        ax.hist(df_samples[col].values, bins=bins)
        ax.set_xlabel(col)
        ax.set_ylabel("Frecuencia")
        plt.show()

## 2. Carga de datos (versión limpia del Lab 1)

In [ ]:
# ======================
# 2) Carga de datos
# ======================

# TODO: Cambia esta ruta a tu dataset limpio
DATA_PATH = "data/alpeshearth_clean.csv"   # ejemplo

# TODO: Cambia el nombre de la columna objetivo
TARGET_COL = "target"  # ejemplo

# (Opcional) columnas a excluir (IDs, leakage, etc.)
DROP_COLS = []  # ejemplo: ["patient_id"]

# Cargar
df = pd.read_csv(DATA_PATH)

# Chequeos mínimos
print("Shape:", df.shape)
display(df.head())

assert TARGET_COL in df.columns, f"No encuentro la columna objetivo: {TARGET_COL}"

## 3. Split Train/Test (test congelado)

In [ ]:
# ======================
# 3) Train/Test split
# ======================

X = df.drop(columns=[TARGET_COL] + DROP_COLS, errors="ignore")
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)

## 4. Esquema de validación cruzada y scoring

In [ ]:
# ======================
# 4) CV + scoring
# ======================

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Usamos scoring múltiple y refit por RMSE (minimizar).
# En sklearn, RMSE se define como 'neg_root_mean_squared_error' (se maximiza),
# así que usaremos su negativo (más grande = mejor) y luego convertiremos a RMSE positivo en la tabla.
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
}

## 5. Actividad 1 — Regresión polinomial (GridSearchCV)

In [ ]:
# ======================
# 5) Actividad 1: Polinomial
# ======================

poly_pipe = Pipeline(steps=[
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", "passthrough"),
    ("model", LinearRegression())
])

param_grid_poly = {
    "poly__degree": [1, 2, 3, 4, 5],
    "scaler": ["passthrough", StandardScaler(), MinMaxScaler()],
}

gs_poly = GridSearchCV(
    estimator=poly_pipe,
    param_grid=param_grid_poly,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    n_jobs=-1,
    return_train_score=False,
)

gs_poly.fit(X_train, y_train)

print("Mejores hiperparámetros (polinomial):", gs_poly.best_params_)
print("Mejor score CV (neg RMSE):", gs_poly.best_score_)

In [ ]:
# Tabla de resultados (conversión a métricas positivas para RMSE y MAE)
tbl_poly = pd.DataFrame(gs_poly.cv_results_)
tbl_poly["mean_test_RMSE"] = -tbl_poly["mean_test_RMSE"]
tbl_poly["std_test_RMSE"]  = tbl_poly["std_test_RMSE"]
tbl_poly["mean_test_MAE"]  = -tbl_poly["mean_test_MAE"]
tbl_poly["std_test_MAE"]   = tbl_poly["std_test_MAE"]
tbl_poly["mean_test_R2"]   = tbl_poly["mean_test_R2"]
tbl_poly["std_test_R2"]    = tbl_poly["std_test_R2"]

# Ranking por RMSE (menor es mejor)
tbl_poly["rank_test_RMSE"] = tbl_poly["mean_test_RMSE"].rank(method="min")

# Tabla simplificada
keep = [c for c in tbl_poly.columns if c.startswith("param_")] + [
    "mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2","rank_test_RMSE"
]
tbl_poly_slim = tbl_poly[keep].sort_values("mean_test_RMSE").reset_index(drop=True)

display(tbl_poly_slim.head(10))

### Evaluación en test (solo para reporte; NO usar para elegir hiperparámetros)

In [ ]:
best_poly = gs_poly.best_estimator_
pred_test = best_poly.predict(X_test)
metrics_poly_test = compute_metrics(y_test, pred_test)

print("Métricas en TEST (polinomial mejor por CV):", metrics_poly_test)

## 6. Actividad 2 — Curva de validación por grado (polinomial)

In [ ]:
# ======================
# 6) Actividad 2: Curva de validación
# ======================

# Agregamos por grado (promedio sobre escaladores y demás combinaciones) o escogemos un escalador fijo.
# En laboratorios, suele ser más interpretable fijar el escalador (p.ej. StandardScaler) y variar degree.

scaler_choice = StandardScaler()  # TODO: si prefieres otra, cámbiala

poly_pipe_fixed = Pipeline(steps=[
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", scaler_choice),
    ("model", LinearRegression())
])

param_grid_degree = {
    "poly__degree": [1, 2, 3, 4, 5, 6, 7],
}

gs_degree = GridSearchCV(
    estimator=poly_pipe_fixed,
    param_grid=param_grid_degree,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,  # útil si quieres ver train vs CV
)

gs_degree.fit(X_train, y_train)

df_deg = pd.DataFrame(gs_degree.cv_results_)
df_deg["mean_test_RMSE"] = -df_deg["mean_test_RMSE"]
df_deg["mean_train_RMSE"] = -df_deg["mean_train_RMSE"]
df_deg["mean_test_MAE"] = -df_deg["mean_test_MAE"]
df_deg["mean_train_MAE"] = -df_deg["mean_train_MAE"]

display(df_deg[["param_poly__degree","mean_train_RMSE","mean_test_RMSE","std_test_RMSE"]].sort_values("param_poly__degree"))

In [ ]:
# Gráfica train vs CV para RMSE
x = df_deg["param_poly__degree"].astype(int).values
rmse_train = df_deg["mean_train_RMSE"].values
rmse_cv = df_deg["mean_test_RMSE"].values
rmse_cv_std = df_deg["std_test_RMSE"].values

fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_title("Curva de validación — RMSE vs grado (polinomial)")
ax.plot(x, rmse_train, marker="o", label="Train mean RMSE")
ax.plot(x, rmse_cv, marker="o", label="CV mean RMSE")
ax.fill_between(x, rmse_cv - rmse_cv_std, rmse_cv + rmse_cv_std, alpha=0.2, label="±1 std (CV)")
ax.set_xlabel("degree")
ax.set_ylabel("RMSE")
ax.legend()
plt.show()

## 7. Actividad 3 — Modelos regularizados: Ridge y Lasso (GridSearchCV)

In [ ]:
# ======================
# 7) Actividad 3: Ridge
# ======================

ridge_pipe = Pipeline(steps=[
    ("scaler", "passthrough"),
    ("model", Ridge(random_state=RANDOM_STATE))
])

alphas = np.logspace(-4, 3, 20)

param_grid_ridge = {
    "scaler": ["passthrough", StandardScaler(), MinMaxScaler()],
    "model__alpha": alphas
}

gs_ridge = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=param_grid_ridge,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    n_jobs=-1,
)

gs_ridge.fit(X_train, y_train)

print("Mejores hiperparámetros (Ridge):", gs_ridge.best_params_)

In [ ]:
tbl_ridge = pd.DataFrame(gs_ridge.cv_results_)
tbl_ridge["mean_test_RMSE"] = -tbl_ridge["mean_test_RMSE"]
tbl_ridge["mean_test_MAE"] = -tbl_ridge["mean_test_MAE"]

keep = [c for c in tbl_ridge.columns if c.startswith("param_")] + [
    "mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2"
]
tbl_ridge_slim = tbl_ridge[keep].sort_values("mean_test_RMSE").reset_index(drop=True)
display(tbl_ridge_slim.head(10))

# Test (solo reporte)
best_ridge = gs_ridge.best_estimator_
metrics_ridge_test = compute_metrics(y_test, best_ridge.predict(X_test))
print("Métricas en TEST (Ridge mejor por CV):", metrics_ridge_test)

In [ ]:
# ======================
# 7) Actividad 3: Lasso
# ======================

lasso_pipe = Pipeline(steps=[
    ("scaler", "passthrough"),
    ("model", Lasso(max_iter=10000, random_state=RANDOM_STATE))
])

param_grid_lasso = {
    "scaler": ["passthrough", StandardScaler(), MinMaxScaler()],
    "model__alpha": alphas
}

gs_lasso = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=param_grid_lasso,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    n_jobs=-1,
)

gs_lasso.fit(X_train, y_train)

print("Mejores hiperparámetros (Lasso):", gs_lasso.best_params_)

In [ ]:
tbl_lasso = pd.DataFrame(gs_lasso.cv_results_)
tbl_lasso["mean_test_RMSE"] = -tbl_lasso["mean_test_RMSE"]
tbl_lasso["mean_test_MAE"] = -tbl_lasso["mean_test_MAE"]

keep = [c for c in tbl_lasso.columns if c.startswith("param_")] + [
    "mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2"
]
tbl_lasso_slim = tbl_lasso[keep].sort_values("mean_test_RMSE").reset_index(drop=True)
display(tbl_lasso_slim.head(10))

# Test (solo reporte)
best_lasso = gs_lasso.best_estimator_
metrics_lasso_test = compute_metrics(y_test, best_lasso.predict(X_test))
print("Métricas en TEST (Lasso mejor por CV):", metrics_lasso_test)

### Variables seleccionadas por Lasso (coeficientes = 0)

In [ ]:
# Nota: para que los coeficientes tengan sentido, revisa si tu dataset requiere escalamiento.
# Extraemos coeficientes del mejor Lasso
lasso_model = best_lasso.named_steps["model"]
coef = pd.Series(lasso_model.coef_, index=X_train.columns).sort_values(key=np.abs, ascending=False)

display(coef.head(20))

zeros = (coef == 0).sum()
print(f"Coeficientes exactamente en cero: {zeros} / {len(coef)}")

## 8. Actividad 4 — Polinomial regularizado (Ridge o Lasso)

In [ ]:
# ======================
# 8) Actividad 4: Polinomial + Ridge (recomendado por estabilidad)
# ======================

poly_ridge_pipe = Pipeline(steps=[
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", "passthrough"),
    ("model", Ridge(random_state=RANDOM_STATE))
])

param_grid_poly_ridge = {
    "poly__degree": [1, 2, 3, 4, 5],
    "scaler": ["passthrough", StandardScaler(), MinMaxScaler()],
    "model__alpha": np.logspace(-4, 3, 15),
}

gs_poly_ridge = GridSearchCV(
    estimator=poly_ridge_pipe,
    param_grid=param_grid_poly_ridge,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    n_jobs=-1,
)

gs_poly_ridge.fit(X_train, y_train)

print("Mejores hiperparámetros (Polinomial + Ridge):", gs_poly_ridge.best_params_)

In [ ]:
tbl_poly_ridge = pd.DataFrame(gs_poly_ridge.cv_results_)
tbl_poly_ridge["mean_test_RMSE"] = -tbl_poly_ridge["mean_test_RMSE"]
tbl_poly_ridge["mean_test_MAE"]  = -tbl_poly_ridge["mean_test_MAE"]

keep = [c for c in tbl_poly_ridge.columns if c.startswith("param_")] + [
    "mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2"
]
tbl_poly_ridge_slim = tbl_poly_ridge[keep].sort_values("mean_test_RMSE").reset_index(drop=True)
display(tbl_poly_ridge_slim.head(10))

best_poly_ridge = gs_poly_ridge.best_estimator_
metrics_poly_ridge_test = compute_metrics(y_test, best_poly_ridge.predict(X_test))
print("Métricas en TEST (Polinomial+Ridge mejor por CV):", metrics_poly_ridge_test)

## 9. Actividad 5 — Comparación y selección del mejor modelo (desempeño + estabilidad)

In [ ]:
# ======================
# 9) Comparación consolidada
# ======================

def best_cv_summary(name: str, gs: GridSearchCV) -> Dict:
    # Extraer mejores métricas promedio y std de CV (en escala positiva)
    res = pd.DataFrame(gs.cv_results_)
    idx = gs.best_index_
    row = res.loc[idx]
    return {
        "Modelo": name,
        "BestParams": gs.best_params_,
        "CV_RMSE_mean": -row["mean_test_RMSE"],
        "CV_RMSE_std": row["std_test_RMSE"],
        "CV_MAE_mean": -row["mean_test_MAE"],
        "CV_MAE_std": row["std_test_MAE"],
        "CV_R2_mean": row["mean_test_R2"],
        "CV_R2_std": row["std_test_R2"],
    }

summary = []
summary.append(best_cv_summary("Polinomial", gs_poly))
summary.append(best_cv_summary("Ridge", gs_ridge))
summary.append(best_cv_summary("Lasso", gs_lasso))
summary.append(best_cv_summary("Polinomial+Ridge", gs_poly_ridge))

df_compare = pd.DataFrame(summary).sort_values("CV_RMSE_mean").reset_index(drop=True)
display(df_compare)

# ======================
# Selección del modelo final (heurística sugerida)
# ======================
# Regla típica: menor RMSE promedio y, en empate cercano, menor std.
# TODO: argumenta tu selección en texto en la siguiente celda.
best_row = df_compare.iloc[0]
print("Candidato por RMSE medio:", best_row["Modelo"])

### Argumento de selección (escribir aquí en texto)
Incluye explícitamente: (i) desempeño promedio en CV (RMSE), (ii) desviación estándar, (iii) complejidad (grado/alpha), (iv) interpretabilidad (Lasso vs Ridge), y (v) coherencia CV vs TEST.

## 10. Actividad 6 — Bootstrapping: intervalos de confianza al 95% en TEST

In [ ]:
# ======================
# 10) Entrenar modelo final en TRAIN completo y bootstrap en TEST
# ======================

# TODO: selecciona explícitamente cuál modelo final usar
# Por defecto, usamos el mejor por CV_RMSE_mean (arriba).
model_map = {
    "Polinomial": gs_poly.best_estimator_,
    "Ridge": gs_ridge.best_estimator_,
    "Lasso": gs_lasso.best_estimator_,
    "Polinomial+Ridge": gs_poly_ridge.best_estimator_,
}

final_model_name = best_row["Modelo"]
final_model = model_map[final_model_name]

# Reentrenar en todo el train
final_model.fit(X_train, y_train)

# Test metrics puntuales
test_pred = final_model.predict(X_test)
test_metrics = compute_metrics(y_test, test_pred)
print("Modelo final:", final_model_name)
print("Métricas puntuales en TEST:", test_metrics)

# Bootstrap
df_samples, df_ci = bootstrap_ci_metrics(final_model, X_test, y_test, n_boot=500, ci=0.95)
display(df_ci)

# Gráficas
plot_bootstrap_distributions(df_samples, bins=30)

## 11. Actividad 7 — Análisis de resultados (responder preguntas)

### 11.1 Análisis cuantitativo
Responde, con evidencia (tablas/gráficas), al menos:
- ¿Cuál modelo obtuvo el mejor desempeño en test?
- ¿Coincide con el mejor promedio en CV? ¿por qué sí/no?
- ¿Por qué la desviación estándar importa?
- ¿Dónde se observa sobreajuste en la curva de validación?
- ¿Qué efecto tuvo la regularización sobre coeficientes?
- ¿Qué indican los IC bootstrap sobre estabilidad?

### 11.2 Análisis cualitativo
- Variables más relevantes según Lasso.
- Interpretación práctica de coeficientes (si aplica al contexto del riesgo).
- Precisión vs interpretabilidad.
- Valor empresarial vs complejidad.

### 11.3 Reflexión conceptual
- Complejidad vs generalización vs estabilidad.
- Fuentes potenciales de sesgo.
- Efecto esperado de aumentar tamaño de muestra.

## 12. Uso de herramientas de IA generativa (obligatorio)

Incluye:
- Herramienta usada y para qué (conceptos, esqueleto de código, depuración, redacción).
- Prompts principales (textuales o resumidos).
- Análisis crítico (qué fue útil/qué corregiste).
- Aportes propios (decisiones, cambios, aprendizaje).

## 13. Conclusiones

Cierra con 2–3 párrafos o bullets: modelo final, hallazgos sobre complejidad, estabilidad y recomendaciones para AlpesHearth.